[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Pruebas automatizadas en *CI/CD* con *pytest*, *coverage* y *tox*.

En este curso las pruebas automatizadas dejan de ser un ejemplo accesorio y se convierten en una condición mínima para empaquetar y desplegar.

## Objetivos.

- Entender por qué las pruebas son un control crítico en el *SDLC*.
- Integrar *pytest* y *coverage* como *quality gates* en *CI*.
- Usar *tox* para validar en múltiples entornos de Python.
- Relacionar pruebas, *checks* y reglas de *merge* en *pipelines* de *CI/CD*.

## Por qué las pruebas son críticas en el *SDLC*.

En un ciclo de vida de software (*SDLC*), las pruebas no son una fase final: son un control continuo de riesgo. Cada cambio de código puede introducir regresiones funcionales, defectos de seguridad o degradaciones de rendimiento.

Sin pruebas automatizadas, el costo de un error crece de forma no lineal: detectar un bug en desarrollo cuesta poco; encontrarlo en producción implica incidentes, rollback, pérdida de confianza y tiempo operativo del equipo.

En un flujo *CI/CD*, las pruebas son un *quality gate* que protege la rama principal y evita promover artefactos defectuosos a *staging* o producción.

## Estrategia de pruebas en *pipelines*.

Un *pipeline* robusto combina distintos tipos de prueba, cada uno con objetivo distinto:

- unitarias: validan lógica aislada y son las más rápidas.
- integración: validan contratos entre módulos, base de datos o *APIs*.
- extremo a extremo: validan flujos reales de negocio.

La práctica recomendada es ejecutar primero lo más rápido y determinista (unitarias), y luego escalar a pruebas más costosas. Así, el *feedback* al desarrollador llega antes y el uso de *runners* es más eficiente.

## El valor de probar en runners efímeros.

Cada ejecución del *pipeline* parte de un entorno limpio. Eso obliga a declarar dependencias, datos y pasos de preparación de forma explícita. La ventaja es que reduce diferencias entre lo que funciona localmente y lo que realmente es desplegable.

Un *job* de pruebas bien construido no solo ejecuta `pytest`; también deja evidencia clara en los *checks* del *pull request* y ayuda a decidir si un *merge* es aceptable.

## *coverage*: medir qué tan bien prueban tus *tests*.

No basta con que `pytest` pase. También importa saber qué porcentaje del código fue ejercitado por las pruebas. Para ello se usa *coverage* (habitualmente mediante [*pytest-cov*](https://pytest-cov.readthedocs.io/)).

Comandos típicos:

```bash
pytest --cov=app --cov-report=term-missing
pytest --cov=app --cov-fail-under=80
```

`--cov-fail-under` convierte la cobertura mínima en una regla automatizada del *pipeline*. No garantiza ausencia de *bugs*, pero evita zonas de código sin validar y mejora disciplina de pruebas a lo largo del tiempo.

## *tox*: pruebas reproducibles en múltiples entornos.

[*tox*](https://tox.wiki/) permite ejecutar la misma suite de pruebas en varias versiones de *Python* y con comandos estandarizados. Es útil cuando quieres validar compatibilidad (por ejemplo 3.10, 3.11 y 3.12) sin duplicar scripts.

Ejemplo mínimo de `tox.ini`:

```ini
[tox]
envlist = py310,py311,py312

[testenv]
deps =
    pytest
    pytest-cov
commands =
    pytest --cov=app --cov-fail-under=80
```

En *CI/CD*, *tox* reduce diferencias entre ejecución local y *runner* remoto, porque obliga a declarar de forma explícita dependencias y comandos por entorno.

## Sobre *nose*: contexto histórico y decisión práctica.

[*nose*](https://nose.readthedocs.io/en/latest/) fue popular hace años, pero hoy se considera legado y sin mantenimiento activo para proyectos nuevos. En ecosistemas modernos, la recomendación es usar *pytest* como base y complementar con *coverage* y, cuando aplica, *tox*.

Si trabajas sobre un repositorio antiguo con *nose*, el enfoque sano es planear una migración gradual a *pytest* para mejorar soporte, plugins y mantenibilidad del pipeline.

## Ejemplo de *quality gate* de pruebas en *GitHub Actions*.

```yaml
jobs:
  tests:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: pip install pytest pytest-cov tox
      - run: pytest --cov=app --cov-fail-under=80
```

Este *gate* asegura tres cosas antes del *merge*: suite funcional verde, umbral mínimo de cobertura y evidencia trazable en los *checks* del repositorio.

## Demo práctica en CLI.

La ejecución práctica de pruebas se encuentra en en [demos/08](demos/08).

Contenido de la demo:

- módulo de ejemplo en [demos/08/calculadora.py](demos/08/calculadora.py).
- pruebas en [demos/08/tests/test_calculadora.py](demos/08/tests/test_calculadora.py).
- guía de ejecución en [demos/08/README.md](demos/08/README.md).

Comando recomendado desde terminal:

```
pytest -q
```
Resultado esperado en este entorno: 7 pruebas correctas.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>